# Trabajo Práctico: Inferencia Estadística Aplicada al Fútbol

En este trabajo práctico utilizaremos la estadística inferencial para responder preguntas clave sobre el rendimiento deportivo en el fútbol.

### Dataset Unificado:
Para todas las pruebas utilizaremos un mismo conjunto de datos simulado (`df_futbol`). Este DataFrame contiene información de 120 partidos con las siguientes columnas:
- `goles_local`: Goles marcados por el equipo jugando en casa.
- `goles_visitante`: Goles marcados por el equipo jugando fuera.
- `goles_t1`: Goles anotados por partido bajo la dirección del *Entrenador 1*.
- `goles_t2`: Goles anotados por partido bajo la dirección del *Entrenador 2*.
- `liga`: Liga a la que pertenece el partido (`LaLiga`, `Premier`, `SerieA`).
- `resultado`: Resultado final desde la perspectiva del local (`Ganó Local`, `Empate`, `Ganó Visitante`).

---
## Bloque Inicial: Preparación del Dataset Común
Ejecuta la siguiente celda para inicializar los datos con los que trabajarás en todo el TP.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# Fijar semilla para reproducibilidad
np.random.seed(42)

n_partidos = 120
data = {
    'goles_local': np.random.poisson(lam=1.6, size=n_partidos),
    'goles_visitante': np.random.poisson(lam=1.1, size=n_partidos),
    'goles_t1': np.random.poisson(lam=1.2, size=n_partidos),
    'goles_t2': np.random.poisson(lam=1.5, size=n_partidos),
    'liga': np.random.choice(['LaLiga', 'Premier', 'SerieA'], size=n_partidos),
    'resultado': np.random.choice(['Ganó Local', 'Empate', 'Ganó Visitante'],
                                   size=n_partidos, p=[0.45, 0.25, 0.30])
}
df_futbol = pd.DataFrame(data)
print("¡Dataset cargado con éxito! Primeros 5 registros:")
df_futbol.head()

¡Dataset cargado con éxito! Primeros 5 registros:


,goles_local,goles_visitante,goles_t1,goles_t2,liga,resultado
0,3,0,2,1,Premier,Ganó Visitante
1,0,0,1,1,SerieA,Ganó Local
2,0,2,0,0,Premier,Ganó Visitante
3,0,0,0,1,Premier,Ganó Visitante
4,3,3,0,2,Premier,Empate


### Ejercicio 1: Estimación puntual
**Pregunta:** ¿Cuántos goles marca en promedio un equipo por partido de local?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Utiliza el método `.mean()` de pandas sobre la columna correspondiente.

In [ ]:
# H0: no aplica (estimación puntual, no hay hipótesis)
# H1: no aplica

media_local = df_futbol['goles_local'].mean()
print(f'Estimación puntual — Media de goles de local: {media_local:.4f}')
print()
print('Interpretación futbolística: en promedio cada equipo local convierte'
      f' {media_local:.2f} goles por partido. Ese único valor es nuestra mejor'
      ' estimación del parámetro poblacional basada en la muestra.')


### Ejercicio 2: Intervalo de confianza
**Pregunta:** ¿Cuál es el rango plausible (IC 95%) para el promedio real de goles de local?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Utiliza `stats.t.interval()` calculando el error estándar de la media (SEM).

In [ ]:
from scipy import stats

goles_local = df_futbol['goles_local']
n     = len(goles_local)
media = goles_local.mean()
sem   = goles_local.std(ddof=1) / np.sqrt(n)

ic = stats.t.interval(0.95, df=n-1, loc=media, scale=sem)

print(f'Media muestral : {media:.4f}')
print(f'IC 95%         : ({ic[0]:.4f}, {ic[1]:.4f})')
print()
print('Interpretación futbolística: con 95% de confianza el promedio real de'
      f' goles de local en la liga está entre {ic[0]:.2f} y {ic[1]:.2f} goles.'
      ' Este rango es la incertidumbre que rodea nuestra estimación puntual.')


### Ejercicio 3: Test t (1 muestra)
**Pregunta:** ¿El equipo anota en promedio exactamente 2 goles por partido de local?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Compara la muestra contra el parámetro `popmean=2.0` usando `stats.ttest_1samp`.

In [ ]:
# H0: mu = 2.0  (el equipo anota exactamente 2 goles de local en promedio)
# H1: mu != 2.0 (el promedio real es distinto de 2)

t_stat, p_val = stats.ttest_1samp(df_futbol['goles_local'], popmean=2.0)

print(f't = {t_stat:.4f}')
print(f'p-valor = {p_val:.4f}')
print()
if p_val < 0.05:
    print('Decisión: Se RECHAZA H0. El promedio de goles de local difiere significativamente de 2.')
else:
    print('Decisión: No se rechaza H0. No hay evidencia para afirmar que la media difiere de 2.')
print()
print('Interpretación futbolística: el rendimiento goleador del local no es exactamente'
      ' 2 goles por partido; el valor real parece ser menor, más cercano a la media muestral.')


### Ejercicio 4: Test t (2 muestras)
**Pregunta:** ¿Anota más goles de local que de visitante?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Considera las muestras de goles locales y visitantes como independientes con `stats.ttest_ind`.

In [ ]:
# H0: mu_local = mu_visitante  (no hay diferencia en el promedio de goles)
# H1: mu_local != mu_visitante (hay diferencia)

t_stat, p_val = stats.ttest_ind(
    df_futbol['goles_local'],
    df_futbol['goles_visitante']
)

print(f'Media goles local    : {df_futbol["goles_local"].mean():.4f}')
print(f'Media goles visitante: {df_futbol["goles_visitante"].mean():.4f}')
print(f't = {t_stat:.4f},  p-valor = {p_val:.4f}')
print()
if p_val < 0.05:
    print('Decisión: Se RECHAZA H0. Hay diferencia significativa: el local anota más.')
else:
    print('Decisión: No se rechaza H0. No hay diferencia significativa entre goles local y visitante.')
print()
print('Interpretación futbolística: este resultado refleja (o no) la conocida ventaja de'
      ' jugar de local, que en datos reales suele ser estadísticamente significativa.')


### Ejercicio 5: Test t pareado
**Pregunta:** ¿Mejoró el rendimiento goleador tras cambiar de entrenador (de T1 a T2)?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Al evaluar al mismo equipo antes y después, las muestras están relacionadas. Usa `stats.ttest_rel`.

In [ ]:
# H0: mu_T2 = mu_T1  (el cambio de entrenador no mejoró el rendimiento)
# H1: mu_T2 > mu_T1  (T2 hace anotar más goles que T1)

t_stat, p_val = stats.ttest_rel(df_futbol['goles_t2'], df_futbol['goles_t1'])

print(f'Media goles T1: {df_futbol["goles_t1"].mean():.4f}')
print(f'Media goles T2: {df_futbol["goles_t2"].mean():.4f}')
print(f't = {t_stat:.4f},  p-valor = {p_val:.4f}')
print()
if p_val < 0.05:
    print('Decisión: Se RECHAZA H0. El cambio de entrenador produjo una mejora significativa.')
else:
    print('Decisión: No se rechaza H0. No hay evidencia de mejora significativa con el nuevo entrenador.')
print()
print('Interpretación futbolística: se usa test pareado porque comparamos el MISMO equipo'
      ' antes y después del cambio — cada partido bajo T1 tiene su par correspondiente bajo T2.')


### Ejercicio 6: ANOVA
**Pregunta:** ¿Las tres ligas (LaLiga, Premier, SerieA) tienen el mismo promedio de goles de local?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Filtra la columna `goles_local` según la `liga` y compáralas mediante `stats.f_oneway`.

In [ ]:
# H0: mu_LaLiga = mu_Premier = mu_SerieA  (las tres ligas tienen el mismo promedio de goles)
# H1: al menos una liga difiere de las demás

laliga  = df_futbol[df_futbol['liga'] == 'LaLiga']['goles_local']
premier = df_futbol[df_futbol['liga'] == 'Premier']['goles_local']
seriea  = df_futbol[df_futbol['liga'] == 'SerieA']['goles_local']

f_stat, p_val = stats.f_oneway(laliga, premier, seriea)

print(f'Medias: LaLiga={laliga.mean():.2f}  Premier={premier.mean():.2f}  SerieA={seriea.mean():.2f}')
print(f'F = {f_stat:.4f},  p-valor = {p_val:.4f}')
print()
if p_val < 0.05:
    print('Decisión: Se RECHAZA H0. Hay diferencia significativa entre al menos un par de ligas.')
else:
    print('Decisión: No se rechaza H0. No hay diferencia significativa entre ligas.')
print()
print('Interpretación futbolística: ANOVA compara los promedios de goles de local entre tres'
      ' ligas simultáneamente sin inflar el error de tipo I al hacer múltiples comparaciones.')


### Ejercicio 7: Mann–Whitney
**Pregunta:** ¿Dos ligas difieren cuando los datos son muy asimétricos o no normales?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Aplica la alternativa no paramétrica `stats.mannwhitneyu` para comparar 'LaLiga' y 'Premier'.

In [ ]:
# H0: LaLiga y Premier tienen la misma distribución de goles de local
# H1: las distribuciones difieren

laliga  = df_futbol[df_futbol['liga'] == 'LaLiga']['goles_local']
premier = df_futbol[df_futbol['liga'] == 'Premier']['goles_local']

u_stat, p_val = stats.mannwhitneyu(laliga, premier, alternative='two-sided')

print(f'U = {u_stat:.4f},  p-valor = {p_val:.4f}')
print()
if p_val < 0.05:
    print('Decisión: Se RECHAZA H0. Diferencia significativa entre LaLiga y Premier.')
else:
    print('Decisión: No se rechaza H0. No hay diferencia significativa entre LaLiga y Premier.')
print()
print('Interpretación futbolística: Mann-Whitney es preferible al t-test cuando los goles'
      ' (datos de conteo) no son normales. Compara rangos en lugar de medias.')


### Ejercicio 8: Kruskal–Wallis
**Pregunta:** ¿Las tres ligas difieren en su distribución sin asumir normalidad?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Extiende la comparación no paramétrica a los tres grupos usando `stats.kruskal`.

In [ ]:
# H0: las tres ligas tienen la misma distribución de goles
# H1: al menos una liga difiere en su distribución

laliga  = df_futbol[df_futbol['liga'] == 'LaLiga']['goles_local']
premier = df_futbol[df_futbol['liga'] == 'Premier']['goles_local']
seriea  = df_futbol[df_futbol['liga'] == 'SerieA']['goles_local']

h_stat, p_val = stats.kruskal(laliga, premier, seriea)

print(f'H = {h_stat:.4f},  p-valor = {p_val:.4f}')
print()
if p_val < 0.05:
    print('Decisión: Se RECHAZA H0. Al menos una liga difiere (no paramétrico).')
else:
    print('Decisión: No se rechaza H0. No hay diferencia entre las tres ligas (no paramétrico).')
print()
print('Interpretación futbolística: Kruskal-Wallis es el equivalente no paramétrico'
      ' del ANOVA; válido aquí porque los goles siguen distribución Poisson, no Normal.')


### Ejercicio 9: Chi-cuadrado
**Pregunta:** ¿La condición de ser local está asociada con ganar el partido?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Construye primero una tabla de contingencia con `pd.crosstab()` usando las columnas 'liga' o 'resultado' y luego aplica `stats.chi2_contingency`.

In [ ]:
# H0: la liga y el resultado del partido son independientes
# H1: la liga está asociada con el resultado

tabla_contingencia = pd.crosstab(df_futbol['liga'], df_futbol['resultado'])
print(tabla_contingencia)
print()

chi2, p_val, dof, esperado = stats.chi2_contingency(tabla_contingencia)

print(f'chi2 = {chi2:.4f},  grados de libertad = {dof},  p-valor = {p_val:.4f}')
print()
if p_val < 0.05:
    print('Decisión: Se RECHAZA H0. La liga está asociada con el resultado del partido.')
else:
    print('Decisión: No se rechaza H0. La liga y el resultado son independientes.')
print()
print('Interpretación futbolística: si hay asociación, algunos resultados son más'
      ' frecuentes en ciertas ligas (ej. más empates en SerieA).')


### Ejercicio 10: Shapiro–Wilk
**Pregunta:** ¿Los goles de local siguen una distribución normal?

**Estructura del Análisis:**
- **Hipótesis:** (Escribe aquí $H_0$ y $H_1$ si corresponde)
- **Código:** (Completa la celda de abajo)
- **Decisión:** (Indica si rechazas o no $H_0$ en base al valor p)
- **Interpretación futbolística:** (¿Qué significa este resultado en el contexto del juego?)

*Pista:* Pon a prueba la hipótesis de normalidad analítica empleando `stats.shapiro`.

In [ ]:
# H0: los goles de local siguen una distribución normal
# H1: los goles de local NO siguen una distribución normal

stat, p_val = stats.shapiro(df_futbol['goles_local'])

print(f'W = {stat:.4f},  p-valor = {p_val:.4f}')
print()
if p_val < 0.05:
    print('Decisión: Se RECHAZA H0. Los goles de local NO siguen distribución normal.')
else:
    print('Decisión: No se rechaza H0. No hay evidencia contra la normalidad.')
print()
print('Interpretación futbolística: los goles son datos de CONTEO (0, 1, 2, ...)'
      ' y siguen una distribución Poisson, no Normal. Por eso esperamos rechazar'
      ' la normalidad — y debemos usar pruebas no paramétricas (Kruskal, Mann-Whitney).')
